<a href="https://colab.research.google.com/github/1021114Carlos/MIT_MM_Finance/blob/Finance-shop/Courses/Foundation%20of%20modern%20finance%20I/M4_fixed_income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp
import math
from IPython.display import display, Math
import numpy as np
import pandas as pd

(Q1)

In [ ]:
display(Math(r"B_p = \frac{1}{(1 + r_t)^t}"))

In [ ]:
def bootstrap_spot_rates(bonds: pd.DataFrame):

  bonds = bonds.copy().sort_values("T").reset_index(drop=True)
  bonds["coupon"] = bonds["coupon_rate"]*bonds["fv"]

  N = int(bonds["T"].max())
  DF = np.full(N + 1, np.nan)
  spot = np.full(N + 1, np.nan)

  # Boostrap
  for _, row in bonds.iterrows():
    T = int(row["T"])
    price = float(row["price"])
    C = float(row["coupon"])
    FV = float(row["fv"])

    if T == 1:
      # 1- year: price = (FV + C)*DF1
      DF[T] = price/(FV + C)
    else:
      # price = sum_{t=1}^{T-1} C*DF[t] + (FV+C)*DF[T]
      pv_coupons = C*np.nansum(DF[1:T])
      DF[T] = (price - pv_coupons)/(FV + C)

    # Convert DF_T -> spot r_T
    spot[T] = DF[T]**(-1/T) - 1

  bonds["DF"] = bonds["T"].map(lambda t: DF[int(t)])
  bonds["spot"] = bonds["T"].map(lambda t: spot[int(t)])
  return bonds

In [ ]:
bonds = pd.DataFrame({"T": [1, 2, 3], "price": [97.5, 96, 98],
                      "coupon_rate": [0, 0.03, 0.035],
                      "fv": [100, 100, 100],})

out = bootstrap_spot_rates(bonds)
print(out[["T", "price", "coupon_rate", "DF", "spot"]])






(Q2)